### **Dummying Data | Tyler Hilbert | March 31, 2026**

This file serves to dummy the data file to maintain confidentiality of those within the report. The purpose of this file is to remove all information possible to maintain confidentiality, while also allowing the data to be usable. Minimizing the amount of changes to the data (i.e. changing keys) will allow this file to be cut whenever this is used in the field.

*Please Note - in order to maintain student confidentiality, outputs have been cleared from this file*

**Importing Libraries and Pulling Data**

The first step is pulling the libraries that I will use. Since this file is focused on the data, only pandas and numpy will be pulled in. This will allow me to examine and manipulate the data as needed.

The data is from the university's system. Data was pulled from Fall 2022 to Spring 2026. This is due in part to limitations of the university's system, but also allows for review of grades post-COVID.

In [ ]:
#Importing Libraries
import pandas as pd #Imports pandas
import numpy as np #imports numpy

Next is pulling the data in and creating the variable that will be used to manipulate and, eventually, download the dummied file.

*Note - I changed the name of the csv file to something else that is not on my computer. That way I do not accidentally run the script w/o thinking before uploading to GitHub*

In [ ]:
#Pulling in the Data
dum = pd.read_csv("mtgradesfull.csv") #reading the raw data file
dum.head() #running the first five observations to make sure it worked.

**Examining the data and reviewing options for data**

Now that the data is pulled in as a data frame, I am going to use . shape to check the size of it and .keys() to examine what the column headers are. This will help guide us toward what columns need removed, kept, or manipulated.

In [ ]:
#Checking Data Size Pt. 1 (Shape)
dum.shape 

There are a large amount of rows, but that is understandable considering the range of time that is being examined. In the future, I may do a sampling of the data instead of viewing the whole thing. 

In [ ]:
#Checking Data Size Pt. 2 (Keys)
dum.keys()

Now I'm going to check a few different aspects of the data - the logic behind each is as follows:
- Campus - I want to make sure I pulled just one campus when I ran the data request
- Subject - Wanting to make sure that all the subjects I need are caught
- Major - Checking to see the variety of majors in the program - it catches anyone in the course, so expecting it be bulky

In [ ]:
#Checking Data Size Pt. 3 (Campus Count)
campusgroup = dum.groupby("Campus").count()["ID"] #making a variable that groups the campuses by the ID
campusgroup #running it to confirm the number. It's only KC, so no point in keeping it

In [ ]:
#Checking Data Size Pt. 4 (Subject Count)
subjectgroup = dum.groupby("Subject").count()["ID"] #Making a variable that groups the subjects by the ID
subjectgroup #running it to confirm the number. It has all of the subjects I need, but there are some that have way more students than others. Will need to consider options to level out analysis later.

In [ ]:
#Checking Data Size Pt. 5 (Major Count)
majorgroup = dum.groupby("Major").count()["ID"] #making a variable that groups the majors by ID
majorgroup #running it to confirm the numbers. Not surprised at the variety considering the subjects we are looking at.

Now that we have examined the data, I want to put the data into different categories (removed, kept or manipulated). Most of the Removed data is data that can be used to help identify students and/or courses.
- Removed Data:
    * ID - Identifiable student information
    * Name - Identifiable student information
    * Preferred Name - Identifiable student information
    * Course Title Short - Since I am randomizing the course numbers, it does not make sense to keep the course titles as well.
    * CRN - I debated keeping this number, but decided against it. This is more viable for students when registering, not analysis.
    * Course Reference Number - I debated keeping this number, but decided against it. This is more viable for students when registering, not analysis.
    * Online_Program - This just looks at the program of the student. Since this data does not include the delivery method, I figure it would be better to remove it.
    * Registration Status Date - Irrelevant to the purpose of the report - however, it could be useful in another report examining when students register.
    * KSU Email Address - Identifiable student information
    * Phone - Identifiable student information
    * FIRST_MINOR - I worry this may help identify students, so I'm going to drop it. Also, since the data is focused on interventions for majors, minors are not that important to consider.
    * SECOND_MINOR - I worry this may help identify students, so I'm going to drop it. Also, since the data is focused on interventions for majors, minors are not that important to consider.
    * Section - Knowing individual sections is not needed for the report - additionally, it would clutter up the filters later if included (some courses have 50+ sections). If this report included things like instructional method or instructor, this could be more useful to compare how students do in each course over time. That could be a future iteration of the report!
- Kept As-Is Data:
    * Department - I figure this can be maintained as a good filter for later
    * Academic Period - This will be useful for making things like bar charts, strip plots, etc. It will also be good for filtering later
    * Class - This will be a useful filter to have later. Plus it is something that can be used to group different intervention levels.
    * Campus - All courses are taught at the same campus. It won't be useful for current iteration of the report, but could be viable in the future to keep in the logic.
- Manipulated Data:
    * Registration Status - This uses abbreviations, so expanding on them will help clarify for others what they are looking at
    * Subject - This column will be kept, but will be concatanated with the course data
    * Course - This column will be kept, but will be concatanated with the subject data. However, the course numbers will be randomized first.
    * Final Grade - In order to further anonymity, a function will be run to randomize the final grades with the midterm grades.
    * Mid Term Grade - In order to further anonymity, a function will be run to randomize the final grades with the midterm grades.
    * Major - This is what we are really looking at. However, we are concerned with majors in a specific college. I am going to dummy out non-majors by putting them in an "other" major


**Dropping Columns**

Having identified what data should be removed to maintain anonymity, I am going to remove the columns from the dataframe to make a whole new one.

In [ ]:
#Dropping the Columns
dum1 = dum.drop( #Telling pandas we are looking to drop columns and assign it to a new variable
    ["ID","Name","Preferred Name","Course Title Short","CRN","Course Reference Number","Online_Program","Registration Status Date","KSU Email Address","Phone","FIRST_MINOR","SECOND_MINOR", "Section"], #listing out the columns that aren't usable
      axis=1) #This is telling pandas to drop the column, not the row
dum1 #running the first variable to get it to work

**Manipulating the Data**

Now that we are down to just the data we need, we can start manipulating the data to get where we need to go. However, I want to avoid manipulating too much here to avoid issues when this gets moved into a "real" scenario. As such, the only manipulation is going to occur to data that hides anonymity - in this case, grade data, course number and major.

**Randomizing the grade data**

The first act of randomization will be randomizing the grade data. However, there will be some work that needs done in order to ensure the grades make sense (like someone who has a status of withdrawn is shown to have grades). To resolve this, Registration Status, Final Grade and Mid Term Grade will be kept together, randomized, and then reassigned to the table.

**Explaining the shuffling code**

The code below is used to shuffle the grade data code. My understanding of the code can be found here, broken into parts:
- *gradeshuffle =*: this is the variable that we call to hold the shuffled data, as well as plug it in later.
- *dum1[["Registration Status, "Final Grade", "Mid Term Grade]]*: This is calling these three columns together
- *.sample(frac=1)*: This is randomly sampling all of the rows - frac=1 is saying to take all of them (if it was .5, it'd be take half the rows). 
- *reset_index(drop=True)*: This is saying get rid of the old index numbers - if we didn't have drop=True, they'd keep their old ones

After creating the gradeshuffle variable, you then can plug those new values into the place of the old columns by putting dum1["column headers"]

In [ ]:
#Shuffling Grade Data Pt. 1 (The Flawed Way)
# I learned about this after doing some digging online. You can read the source I got this from by visiting here: https://www.geeksforgeeks.org/python/pandas-how-to-shuffle-a-dataframe-rows/
#gradeshuffle = dum1[["Registration Status","Final Grade", "Mid Term Grade"]].sample(frac=1).reset_index(drop=True) #This is creating a group of the Reg Status and grade data, and then randomizing their location
#dum1[["Registration Status","Final Grade", "Mid Term Grade"]] = gradeshuffle # This is replacing the Reg Status and grade data cells with the shuffled data
#dum1 #Running it to see if it works - it does, so that helps add a layer of anonymity

When I was working with the post-shuffled data and cleaning it up, I realized that, when I shuffled the data, it mixed in values from the current term across the terms that already had existing data. This made it seem like the prior terms had instances where the instructor never entered a final grade when one should already exist. This would throw off some future figures that look at how midterm grades relate to final grades. Below is the "corrected" version of the report w/ shuffling. I did use ChatGPT for this part of it, because I wanted to make sure I was doing it right. The prompt I provided it is below:

*I am trying to randomize some data in a pandas dataframe, but I want it to meet a condition (i.e. only do this where the value does not equal another value). How would I rewrite the below code to accomplish this?*

*gradeshuffle = dum1[["Registration Status","Final Grade", "Mid Term Grade"]].sample(frac=1).reset_index(drop=True)*

*dum1[["Registration Status","Final Grade", "Mid Term Grade"]] = gradeshuffle*

*dum1*

In [ ]:
#Shuffling Grade Data Pt. 2 (Variables w/ ChatGPT)
cols = ["Registration Status", "Final Grade", "Mid Term Grade"] #defining the columns I want to keep a hold of
priormask = dum1["Academic Period"] != 202610 #The masking criteria for the terms outside of the current one
currentmask = dum1["Academic Period"] == 202610 #The masking criteria for the term that is ongoing

In [ ]:
#Shuffling Grade Data Pt. 3 (Shuffling non-current terms)
priorshuffled = dum1.loc[priormask, cols].sample(frac=1).reset_index(drop=True) #This is shuffling the data that meet the conditions outlined above, and only grabbing the columns with the impacted variables
dum1.loc[priormask, cols] = priorshuffled.values #Now we are plugging in those shuffled values

In [ ]:
#Shuffling Grade Data Pt. 4 (Shuffling Current Term)
currentshuffled = dum1.loc[currentmask, cols].sample(frac=1).reset_index(drop=True) #This is just shuffling the columns we indicated earlier, but only the current term
dum1.loc[currentmask, cols] = currentshuffled.values

In [ ]:
#Shuffling Grade Data Pt. 5 (Checking if it worked)
dum1.head()

**Randomizing the Course Numbers**

Another randomization I would like to do is the course numbers. While someone not familiar with the university would be unable to tell what the courses are as is, they could still Google it and find the courses. To avoid this, I want to randomize the course numbers with whole new numbers. To avoid having crazy numbers, I want the course numbers to fall in the range of 10000 - 29999.

I had to do some research on the best way to do this. Since there are a lot of course numbers, using the standard map method we used in case would be tedious. I found this site (https://dev.to/codespent/understanding-map-filter-and-zip-in-python-3ifn), and read it through. The zip() approach seems like a good idea - making one variables that is catches all the IDs and using numpy to make randomized numbers before putting it in zip is the way to go.

I included references for the different sections below - that way I have a reference for later.

In [ ]:
#Shuffling Course IDs Pt. 1 (Pulling the Course IDs - https://pandas.pydata.org/docs/reference/api/pandas.unique.html)
courseid = dum1["Course"].unique() #This is creating a variable that pulls all of the unique course numbers from the file. 
courseid #running it to make sure it works

In [ ]:
#Shuffling Course IDs Pt. 2 (Creating a shuffled list - https://numpy.org/doc/2.1/reference/random/generated/numpy.random.randint.html)
shuffledid = np.random.randint(10000,29999, #This is creating a new variable. Using numpy and randint, I am saying creating numbers between the range of 10000-29999 (the number range of courses that have midterms)
                               size=len(courseid)) #This last part is setting the cap on the list. It is saying that the length of the variables should be the same as courseid so it makes only as many as needed
shuffledid #running it to make sure it works

In [ ]:
#Shuffling Course IDs Pt. 3 (Mapping the course numbers - https://www.w3schools.com/python/ref_func_zip.asp)
mapid = dict(zip(courseid,shuffledid)) #My understanding of this is that it is pairing the courseid with the shuffled id. It is saved as a dictionary, and then pairs them together as tuples
mapid #running it to check it out

In [ ]:
#Shuffling Course IDs Pt. 4 (Plugging in the shuffled numbers - https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.map.html)
dum1["Course"] = dum1["Course"].map(mapid) #This is calling the first column. Then, the .map function is saying find the corresponding number in Course, and plug in the corresponding value (i.e. if value in Course is 10000, find it in mapid and plug in the sub value)
dum1

**Removing non-relevant majors**

The last step for dummying the file is only focusing on the core majors. Since there are a variety of majors who have taken these courses (over 180 per the majorgroup variable from earlier), we want to anonymize those students since there could be one-offs enrolled in the courses. Using my knowledge of the programs, I made a list of all the relevant majors that would be active, and opted to replace non-majors with OTH. You can see the code below with explanations.

In [ ]:
#Making an Other Variable Pt. 1 (Creating the list of relevant majors)
coremajors = ["ARCS","ARCH","COMA","ID","ADV","APMD","COMM","DMP","EMAT","JNL","PHOT","PR","UXDE","VCD","ARTE","ARTH","DANC","DNST","FD","FM","MUS","MUED","MUST","MUT","SART","TDTP","THEA"]
#this is the list of majors - no specific order, just based on my documentation

**Breaking down the Non-Major code**

This is an explanation of the below code:
- *dum1["Major"] =*: This is calling the Major column in the dum1 dataframe - this is where the results after the = will get put
- *dum1["Major"].where()*: This is saying in the Major column, look at this situation as described in where (the situation that will be checked is saying if this is true, keep the item or replace it)
- *.where(dum1["Major"].isin(coremajors), "OTH")*: This part is what .where is checking. It is basically saying in the situations where the Major in the Major column is in the coremajors variable, keep it. Where it is false, put in OTH instead

I learned about this via the following sources:
- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.where.html (.where)
- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isin.html (.isin)

Since we are looking at the file as a whole and just blanket replacing the majors, I think this approach works well - once we get to the "main" file, it'll get more complex though since I will need to add a college column that looks at majors by college.


In [ ]:
#Making an Other Variable Pt. 1 (Replacing the other majors)
dum1["Major"] = dum1["Major"].where(dum1["Major"].isin(coremajors), "OTH")

dum1.head()

In [ ]:
#Converting the data to a CSV file
dum1.to_csv("mtgradesanon.csv")

**Wrapping up the dummy file**

I feel like I've replaced everything I wanted to in this file to remove identifiers from the student data. I will write my full reflection in the Milestone 1 file, but some key takeaways from this experience are as below:
- Using variables is great - it helped streamline the process of replacing the other majors. I'll want to pocket that variable for later - it may be useful when I want to introduce a "college" column in future iterations.
- I've slept on numpy - I feel like we didn't use numpy that much in class, so I didn't think much of it. However, it was really useful when using .map to replace the class numbers. I cannot think of situations in which I'll use it professionally, but it may be useful if I ever need to showcase data to third parties.
- The skills learned here could be applied to some of the data work I already do - I currently use Excel for most of my data work. However, I could transition to using pandas for cleaning my data before translating it to PowerBI/Plotly visualizations. I think the frontend work will be a lot though - especially when I get to columns that look at what unit(s) majors belong to (or replacing full majors w/ their abbreviations)

The last bit I'll do is save the cleaned data file and clear outputs. This way I can upload this file to GitHub w/o worry about student data leaking. I am also using nbstripout (https://github.com/kynan/nbstripout) to clean some of the metadata from this file too. That way it is as clean as humanly possible before I upload it to the GitHub.